In [90]:
import pandas as pd
import numpy as np
import time

print(pd.__version__)
print(np.__version__)

3.0.2
2.0.1


#### Basic data structures in pandas
Pandas provides two types of classes for handling data:
- `Series`: a one-dimensional labelled array, holding data of any type.
- `DataFrame`: a two-dimensional data structure that holds data like a two-dimensional array or a table with rows and columns.

#### Object creation
Creating a `Series` by passing a list of values and letting python create a default `RangeIndex`.

In [91]:
# class pandas.Series(data=None, index=None, dtype=None, name=None, copy=None)
s = pd.Series(data=[1, 3, 5, np.nan, 6, 8],
               name="pandas_series",
                 dtype=float)

print(s, type(s), s.dtype, sep=("; ")) # dtype is a float probably because of np.nan

0    1.0
1    3.0
2    5.0
3    NaN
4    6.0
5    8.0
Name: pandas_series, dtype: float64; <class 'pandas.Series'>; float64


Creating a `DataFrame` by passing a NumPy array with a datetime index using `date_range()` and labeled columns:

In [92]:
# create the index (datetime)
dates = pd.date_range(start="2013-01-01", periods=6)

# create the dataframe with the labeled columns
rng = np.random.default_rng(0) # for reproducibility of random values
df = pd.DataFrame(rng.random(24).reshape((6, 4)),
                  index=dates,
                  columns=list("ABCD"),
                  )

print(df)

                   A         B         C         D
2013-01-01  0.636962  0.269787  0.040974  0.016528
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-05  0.863179  0.541461  0.299712  0.422687
2013-01-06  0.028320  0.124283  0.670624  0.647190


Creating a `DataFrame` by passing a dictionary of objects where the keys are the column labels and the values are the column values.

In [93]:
df2 = pd.DataFrame(
    {
        "A": 1.0,
        "B": pd.Timestamp("2013-01-02"),
        "C": pd.Series(1, index=(list(range(4))), dtype=float),
        "D": np.array([3] * 4),
        "E": pd.Categorical(values=["test", "train", "test", "train"]),
        "F": "foo"
    }
)

print(df2)

     A          B    C  D      E    F
0  1.0 2013-01-02  1.0  3   test  foo
1  1.0 2013-01-02  1.0  3  train  foo
2  1.0 2013-01-02  1.0  3   test  foo
3  1.0 2013-01-02  1.0  3  train  foo


In [94]:
print(df2.dtypes)

A           float64
B    datetime64[us]
C           float64
D             int64
E          category
F               str
dtype: object


The columns of the resulting `DataFrame` have different dtypes:

**practice**

In [95]:
invoice = pd.DataFrame(
    {
    "Item_ID": list(range(101, 105)),
    "Category": "Gadget",
    "Base_Price": np.array([19.99, 25.50, 9.99, 49.00]),
    "Stock_Status": pd.Series(data=["In Stock", "Low Stock", "In Stock", "In Stock"]),
    "Audit_Date": pd.Timestamp("20260525")
    }
)

print(invoice)

   Item_ID Category  Base_Price Stock_Status Audit_Date
0      101   Gadget       19.99     In Stock 2026-05-25
1      102   Gadget       25.50    Low Stock 2026-05-25
2      103   Gadget        9.99     In Stock 2026-05-25
3      104   Gadget       49.00     In Stock 2026-05-25


In [96]:
print(invoice.dtypes)

Item_ID                  int64
Category                   str
Base_Price             float64
Stock_Status               str
Audit_Date      datetime64[us]
dtype: object


#### Viewing Data
Use `DataFrame.head()` and `DataFrame.tail()` to view the top and bottom rows of the frame respectively (5 is the default if no value is specified):

In [97]:
print(df.head(3))
print(df.tail(3))

                   A         B         C         D
2013-01-01  0.636962  0.269787  0.040974  0.016528
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739
                   A         B         C         D
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-05  0.863179  0.541461  0.299712  0.422687
2013-01-06  0.028320  0.124283  0.670624  0.647190


Display the `DataFrame.index` or `DataFrame.columns`:

In [98]:
print(df.index)
print(df.columns)

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06'],
              dtype='datetime64[us]', freq='D')
Index(['A', 'B', 'C', 'D'], dtype='str')


Return a NumPy representation of the underlying data with `DataFrame.to_numpy()` without the index or column labels:

In [99]:
print(s.to_numpy(na_value=0)) # na_value replaces the nan value with a specific value

[1. 3. 5. 0. 6. 8.]


NumPy arrays have one dtype for the entire array while pandas DataFrames have one dtype per column.

When you call `DataFrame.to_numpy()`, pandas will find the NumPy dtype that can hold all of the dtypes in the DataFrame. If the common data type is `object`, `DataFrame.to_numpy()` will require copying data.

In [100]:
print(df2.to_numpy(), df2.to_numpy().dtype, sep=(", "))

[[1.0 Timestamp('2013-01-02 00:00:00') 1.0 3 'test' 'foo']
 [1.0 Timestamp('2013-01-02 00:00:00') 1.0 3 'train' 'foo']
 [1.0 Timestamp('2013-01-02 00:00:00') 1.0 3 'test' 'foo']
 [1.0 Timestamp('2013-01-02 00:00:00') 1.0 3 'train' 'foo']], object


`describe()` shows a quick statistic summary of your data:

In [101]:
print(df.describe())

              A         B         C         D
count  6.000000  6.000000  6.000000  6.000000
mean   0.623793  0.469491  0.527242  0.332383
std    0.319052  0.391789  0.296431  0.315579
min    0.028320  0.033586  0.040974  0.002739
25%    0.566959  0.160659  0.376443  0.056310
50%    0.725116  0.405624  0.638630  0.299171
75%    0.846371  0.819932  0.714898  0.591064
max    0.863179  0.935072  0.815854  0.729497


In [102]:
print(df.info())

<class 'pandas.DataFrame'>
DatetimeIndex: 6 entries, 2013-01-01 to 2013-01-06
Freq: D
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A       6 non-null      float64
 1   B       6 non-null      float64
 2   C       6 non-null      float64
 3   D       6 non-null      float64
dtypes: float64(4)
memory usage: 240.0 bytes
None


Transposing your data:

In [103]:
print(df, df.T, sep=("\n"*2))
print(df.T.shape)

                   A         B         C         D
2013-01-01  0.636962  0.269787  0.040974  0.016528
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-05  0.863179  0.541461  0.299712  0.422687
2013-01-06  0.028320  0.124283  0.670624  0.647190

   2013-01-01  2013-01-02  2013-01-03  2013-01-04  2013-01-05  2013-01-06
A    0.636962    0.813270    0.543625    0.857404    0.863179    0.028320
B    0.269787    0.912756    0.935072    0.033586    0.541461    0.124283
C    0.040974    0.606636    0.815854    0.729655    0.299712    0.670624
D    0.016528    0.729497    0.002739    0.175656    0.422687    0.647190
(4, 6)


`DataFrame.sort_index()` sorts by an axis:

In [104]:
# axis=0  --> (row), axis=1 --> (columns)
print(df.sort_index(axis=0, ascending=False))
print(df.sort_index(axis=1, ascending=False))

                   A         B         C         D
2013-01-06  0.028320  0.124283  0.670624  0.647190
2013-01-05  0.863179  0.541461  0.299712  0.422687
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-03  0.543625  0.935072  0.815854  0.002739
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-01  0.636962  0.269787  0.040974  0.016528
                   D         C         B         A
2013-01-01  0.016528  0.040974  0.269787  0.636962
2013-01-02  0.729497  0.606636  0.912756  0.813270
2013-01-03  0.002739  0.815854  0.935072  0.543625
2013-01-04  0.175656  0.729655  0.033586  0.857404
2013-01-05  0.422687  0.299712  0.541461  0.863179
2013-01-06  0.647190  0.670624  0.124283  0.028320


`DataFrame.sort_values()` sorts by values:

In [105]:
print(df)
# sort the entire dataframe with respect to the values in B
print(df.sort_values(by="B")) # ascending is True by default

                   A         B         C         D
2013-01-01  0.636962  0.269787  0.040974  0.016528
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-05  0.863179  0.541461  0.299712  0.422687
2013-01-06  0.028320  0.124283  0.670624  0.647190
                   A         B         C         D
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-06  0.028320  0.124283  0.670624  0.647190
2013-01-01  0.636962  0.269787  0.040974  0.016528
2013-01-05  0.863179  0.541461  0.299712  0.422687
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739


#### Selection

*Getitem ([])*

For a `DataFrame`, passing a single label selects a column and yields a `Series`:

In [106]:
print(df["A"], type(df["A"]), sep="; ")

2013-01-01    0.636962
2013-01-02    0.813270
2013-01-03    0.543625
2013-01-04    0.857404
2013-01-05    0.863179
2013-01-06    0.028320
Freq: D, Name: A, dtype: float64; <class 'pandas.Series'>


If the label (column name) contains only letters, numbers and underscores, you can alternatively use the column name attribute

In [107]:
print(df.A)

2013-01-01    0.636962
2013-01-02    0.813270
2013-01-03    0.543625
2013-01-04    0.857404
2013-01-05    0.863179
2013-01-06    0.028320
Freq: D, Name: A, dtype: float64


Passing a list of column labels selects multiple columns, which can be useful for getting a subset/rearranging:

In [108]:
column_labels = ["A", "B"]
column_labels_ = ["B", "A"] # order  matters
print(df[column_labels], df[column_labels_], sep="\n")

                   A         B
2013-01-01  0.636962  0.269787
2013-01-02  0.813270  0.912756
2013-01-03  0.543625  0.935072
2013-01-04  0.857404  0.033586
2013-01-05  0.863179  0.541461
2013-01-06  0.028320  0.124283
                   B         A
2013-01-01  0.269787  0.636962
2013-01-02  0.912756  0.813270
2013-01-03  0.935072  0.543625
2013-01-04  0.033586  0.857404
2013-01-05  0.541461  0.863179
2013-01-06  0.124283  0.028320


##### Selection by index  
For a `DataFrame`, passing a slice `:` selects **only** matching rows w.r.t. the index:

In [109]:
print(df[0:3])

                   A         B         C         D
2013-01-01  0.636962  0.269787  0.040974  0.016528
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739


In [110]:
print(df["20130104":"20130106"])

                   A         B         C         D
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-05  0.863179  0.541461  0.299712  0.422687
2013-01-06  0.028320  0.124283  0.670624  0.647190


##### Selection by label  
Selecting a row matching a label using `DataFrame.loc()` or `DataFrame.at()`  
- `loc` (location) and `at` refers to accessing (a) value(s) at a specific label-based location

In [111]:
print(dates)
print(df.loc[dates[0]:dates[2], :]) # df.loc[row, column]

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06'],
              dtype='datetime64[us]', freq='D')
                   A         B         C         D
2013-01-01  0.636962  0.269787  0.040974  0.016528
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739


Selecting all rows (:) with a select column labels:

In [112]:
print(df.loc[:, ["A", "B"]])

                   A         B
2013-01-01  0.636962  0.269787
2013-01-02  0.813270  0.912756
2013-01-03  0.543625  0.935072
2013-01-04  0.857404  0.033586
2013-01-05  0.863179  0.541461
2013-01-06  0.028320  0.124283


For label slicing, both endpoints are included:

In [113]:
print(df.loc["20130102":"20130104", ["A","B"]])

                   A         B
2013-01-02  0.813270  0.912756
2013-01-03  0.543625  0.935072
2013-01-04  0.857404  0.033586


Selecting a single row and column label returns a scalar:

In [114]:
print(df.loc[dates[0],"A"])

0.6369616873214543


For getting fast access to a scalar (equivalent to the prior method):  

The main difference between `df.at[]` and `df.loc[]` in pandas is their scope and speed. While both use labels to access data, `at` is optimized for a single value while being significantly faster, whereas `loc` is versatile enough to handle groups of rows and columns.

In [115]:
print(df.at[dates[0], "A"])

0.6369616873214543


##### Selection by position

Select via the position of the passed integers using `df.iloc[]` or `df.iat[]`

In [116]:
print(df, end=("\n\n"))
print(df.iloc[3, :], type(df.iloc[3]), sep="; ") # df.iloc[row, column]

                   A         B         C         D
2013-01-01  0.636962  0.269787  0.040974  0.016528
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-05  0.863179  0.541461  0.299712  0.422687
2013-01-06  0.028320  0.124283  0.670624  0.647190

A    0.857404
B    0.033586
C    0.729655
D    0.175656
Name: 2013-01-04 00:00:00, dtype: float64; <class 'pandas.Series'>


Integer slices acts similar to NumPy/Python:

In [117]:
print(df.iloc[3:5, 0:2]) # endpoint isn't included

                   A         B
2013-01-04  0.857404  0.033586
2013-01-05  0.863179  0.541461


Using a list of integer position locations

In [118]:
print(df.iloc[[1, 2, 4], [0, 2]])

                   A         C
2013-01-02  0.813270  0.606636
2013-01-03  0.543625  0.815854
2013-01-05  0.863179  0.299712


In [119]:
# For slicing rows explicitly
print(df.iloc[1:3, :])
# For slicing columns explicitly
print(df.iloc[:, 1:3])

                   A         B         C         D
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739
                   B         C
2013-01-01  0.269787  0.040974
2013-01-02  0.912756  0.606636
2013-01-03  0.935072  0.815854
2013-01-04  0.033586  0.729655
2013-01-05  0.541461  0.299712
2013-01-06  0.124283  0.670624


In [120]:
# For getting a value (scalar) explicitly:
print(df.iloc[1, 1])
print(df.iat[1, 1]) # faster

0.9127555772777217
0.9127555772777217


##### Boolean indexing  
Select rows where `df.A` is grater than `0`:

In [121]:
print(df["A"] > 0.7)
print()
print(df[df["A"] > 0.7])

2013-01-01    False
2013-01-02     True
2013-01-03    False
2013-01-04     True
2013-01-05     True
2013-01-06    False
Freq: D, Name: A, dtype: bool

                   A         B         C         D
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-05  0.863179  0.541461  0.299712  0.422687


Selecting values from a DataFrame where the condition is met:

In [122]:
print(df)
print()
print(df[df > 0.7])

                   A         B         C         D
2013-01-01  0.636962  0.269787  0.040974  0.016528
2013-01-02  0.813270  0.912756  0.606636  0.729497
2013-01-03  0.543625  0.935072  0.815854  0.002739
2013-01-04  0.857404  0.033586  0.729655  0.175656
2013-01-05  0.863179  0.541461  0.299712  0.422687
2013-01-06  0.028320  0.124283  0.670624  0.647190

                   A         B         C         D
2013-01-01       NaN       NaN       NaN       NaN
2013-01-02  0.813270  0.912756       NaN  0.729497
2013-01-03       NaN  0.935072  0.815854       NaN
2013-01-04  0.857404       NaN  0.729655       NaN
2013-01-05  0.863179       NaN       NaN       NaN
2013-01-06       NaN       NaN       NaN       NaN


Unsing the `isin()` method for filtering:

In [123]:
df2 = df.copy()

df2["E"] = ["one", "one", "two", "three", "four", "three"]
print(df2)

print(df2[df2["E"].isin(["two", "four"])])

                   A         B         C         D      E
2013-01-01  0.636962  0.269787  0.040974  0.016528    one
2013-01-02  0.813270  0.912756  0.606636  0.729497    one
2013-01-03  0.543625  0.935072  0.815854  0.002739    two
2013-01-04  0.857404  0.033586  0.729655  0.175656  three
2013-01-05  0.863179  0.541461  0.299712  0.422687   four
2013-01-06  0.028320  0.124283  0.670624  0.647190  three
                   A         B         C         D     E
2013-01-03  0.543625  0.935072  0.815854  0.002739   two
2013-01-05  0.863179  0.541461  0.299712  0.422687  four


##### Setting  
Seting a new column automatically aligns the data by the indexes:

In [144]:
s1 = pd.Series(
    [1, 2, 3, 4, 5, 6],
    index=pd.date_range("20130102", periods=6)
)
print(s1)

df["F"] = s1
print(df)

2013-01-02    1
2013-01-03    2
2013-01-04    3
2013-01-05    4
2013-01-06    5
2013-01-07    6
Freq: D, dtype: int64
                   A         B         C    D    F
2013-01-01  0.000000  0.000000  0.040974  5.0  NaN
2013-01-02  0.813270  0.912756  0.606636  5.0  1.0
2013-01-03  0.543625  0.935072  0.815854  5.0  2.0
2013-01-04  0.857404  0.033586  0.729655  5.0  3.0
2013-01-05  0.863179  0.541461  0.299712  5.0  4.0
2013-01-06  0.028320  0.124283  0.670624  5.0  5.0


Setting values by label:

In [125]:
df.at[dates[0], "A"] = 0

print(df)

                   A         B         C         D   F
2013-01-01  0.000000  0.269787  0.040974  0.016528 NaN
2013-01-02  0.813270  0.912756  0.606636  0.729497 NaN
2013-01-03  0.543625  0.935072  0.815854  0.002739 NaN
2013-01-04  0.857404  0.033586  0.729655  0.175656 NaN
2013-01-05  0.863179  0.541461  0.299712  0.422687 NaN
2013-01-06  0.028320  0.124283  0.670624  0.647190 NaN


Setting by position:

In [126]:
df.iat[0, 1] = 0

print(df)

                   A         B         C         D   F
2013-01-01  0.000000  0.000000  0.040974  0.016528 NaN
2013-01-02  0.813270  0.912756  0.606636  0.729497 NaN
2013-01-03  0.543625  0.935072  0.815854  0.002739 NaN
2013-01-04  0.857404  0.033586  0.729655  0.175656 NaN
2013-01-05  0.863179  0.541461  0.299712  0.422687 NaN
2013-01-06  0.028320  0.124283  0.670624  0.647190 NaN


Setting by assigning with a NumPy array:

In [127]:
df.loc[:, "D"] = np.array([5] * len(df))

print(df)

                   A         B         C    D   F
2013-01-01  0.000000  0.000000  0.040974  5.0 NaN
2013-01-02  0.813270  0.912756  0.606636  5.0 NaN
2013-01-03  0.543625  0.935072  0.815854  5.0 NaN
2013-01-04  0.857404  0.033586  0.729655  5.0 NaN
2013-01-05  0.863179  0.541461  0.299712  5.0 NaN
2013-01-06  0.028320  0.124283  0.670624  5.0 NaN


A `where` operation with setting:

In [128]:
df2 = df.copy()

df2[df2 > 0.7] = -df2

print(df2)

                   A         B         C    D   F
2013-01-01  0.000000  0.000000  0.040974 -5.0 NaN
2013-01-02 -0.813270 -0.912756  0.606636 -5.0 NaN
2013-01-03  0.543625 -0.935072 -0.815854 -5.0 NaN
2013-01-04 -0.857404  0.033586 -0.729655 -5.0 NaN
2013-01-05 -0.863179  0.541461  0.299712 -5.0 NaN
2013-01-06  0.028320  0.124283  0.670624 -5.0 NaN


#### Missing data  
For NumPy data types, np.nan represents missing data. It is by default not included in computations.

Reindexing allows you to change/add/delete the index on a specified axis. This returns a copy of the data:

In [129]:
print(df)

df1 = df.reindex(
    index=dates[0:4],
    columns = list(df.columns) + ["E"]
)

df1.loc[[dates[0], dates[1]], "E"] = 1

print(df1)

                   A         B         C    D   F
2013-01-01  0.000000  0.000000  0.040974  5.0 NaN
2013-01-02  0.813270  0.912756  0.606636  5.0 NaN
2013-01-03  0.543625  0.935072  0.815854  5.0 NaN
2013-01-04  0.857404  0.033586  0.729655  5.0 NaN
2013-01-05  0.863179  0.541461  0.299712  5.0 NaN
2013-01-06  0.028320  0.124283  0.670624  5.0 NaN
                   A         B         C    D   F    E
2013-01-01  0.000000  0.000000  0.040974  5.0 NaN  1.0
2013-01-02  0.813270  0.912756  0.606636  5.0 NaN  1.0
2013-01-03  0.543625  0.935072  0.815854  5.0 NaN  NaN
2013-01-04  0.857404  0.033586  0.729655  5.0 NaN  NaN


`DataFrame.dropna()` drops any rows that have missing data:

In [130]:
print(df1.dropna(how="any", axis=0))

Empty DataFrame
Columns: [A, B, C, D, F, E]
Index: []


`DataFrame.fillna()` fills missing data:

In [131]:
print(df1.fillna(value=5))

                   A         B         C    D    F    E
2013-01-01  0.000000  0.000000  0.040974  5.0  5.0  1.0
2013-01-02  0.813270  0.912756  0.606636  5.0  5.0  1.0
2013-01-03  0.543625  0.935072  0.815854  5.0  5.0  5.0
2013-01-04  0.857404  0.033586  0.729655  5.0  5.0  5.0


`isna()` gets the boolean mask where values are `nan`:

In [132]:
print(pd.isna(df1))

                A      B      C      D     F      E
2013-01-01  False  False  False  False  True  False
2013-01-02  False  False  False  False  True  False
2013-01-03  False  False  False  False  True   True
2013-01-04  False  False  False  False  True   True


#### Operations

##### Stats  
Operations in general exclude missing data

In [133]:
print(df.mean(axis=0)) # axis=0 gets the mean column-wise

A    0.517633
B    0.424526
C    0.527242
D    5.000000
F         NaN
dtype: float64


In [134]:
print(df.mean(axis=1)) # axis=1 gets the mean row-wise

2013-01-01    1.260243
2013-01-02    1.833165
2013-01-03    1.823638
2013-01-04    1.655161
2013-01-05    1.676088
2013-01-06    1.455807
Freq: D, dtype: float64


Operating with another `Series` or `DataFrame` with a different index or column will align the result with the union of the index or column labels. In addition, pandas automatically broadcasts along the specified dimension and will fill unaligned labels with np.nan.

This is describing the "magic" of how pandas behaves when you do math with two datasets that do not perfectly match up.

If a label exists in Dataset A but not in Dataset B (or vice versa), pandas creates a "union", meaning the final result will contain every label that existed in either dataset. However, because it cannot add a number to a missing value, it fills the gap with NaN (Not a Number).

In [135]:
day1 = pd.Series({"Apples": 10, "Bananas": 5, "Cherries": 20})
day2 = pd.Series({"Bananas": 3, "Cherries": 7, "Dates": 15})

print(day1, day2, day1 + day2, sep=("\n\n"))

Apples      10
Bananas      5
Cherries    20
dtype: int64

Bananas      3
Cherries     7
Dates       15
dtype: int64

Apples       NaN
Bananas      8.0
Cherries    27.0
Dates        NaN
dtype: float64


If you subtract a 1D pandas Series from a 2D pandas DataFrame, pandas will automatically align the Series' index with the DataFrame's columns, and then "broadcast" (apply) that calculation across every single row

In [ ]:
scores = pd.DataFrame({
    "Math": [90, 80],
    "Science": [85, 95]
}, index=["Alice", "Bob"])

curve = pd.Series({"Math": 5, "Science": 10})

print(scores, curve, scores.sub(curve), sep=("\n\n"))

       Math  Science
Alice    90       85
Bob      80       95

Math        5
Science    10
dtype: int64

       Math  Science
Alice    85       75
Bob      75       85


**Mentions**

The `.shift()` method literally pushes all the data in your DataFrame or Series up or down by a specified number of rows. The index (the row labels) stays exactly where it is, but the data slides past it.

Because the data is moving but the DataFrame stays the same size, pandas fills the newly emptied spaces with `NaN`.

In [137]:
sales = pd.Series([100, 150, 130], index=["Mon", "Tue", "Wed"])

# Shift the data down by 1 row
shifted_sales = sales.shift(1)

print(shifted_sales)

Mon      NaN
Tue    100.0
Wed    150.0
dtype: float64


The `.sub()` method stands for subtract. At its most basic level, writing `df1.sub(df2)` does the exact same thing as writing `df1 - df2`.

Why use it instead of the `-` symbol?
Remember how pandas aligns data and fills missing gaps with NaN? The `-` symbol forces you to accept those NaN values. The .`sub()` method, however, gives you access to a superpower parameter: `fill_value`.

In [138]:
print(day1.sub(day2, fill_value=0))

Apples      10.0
Bananas      2.0
Cherries    13.0
Dates      -15.0
dtype: float64


By default, when you subtract a Series from a DataFrame using `df.sub()`, pandas matches the Series index to the DataFrame's columns and applies the math row by row.

When you set `axis="index"` (or axis=0), this behavior flips entirely.

Setting `axis="index"` tells pandas to match the Series index to the DataFrame's index (the row labels), and then "broadcast" (stretch) that math across every single column.

Imagine you have a DataFrame of monthly expenses for different departments:

In [139]:
expenses = pd.DataFrame({
    "Marketing": [1000, 1200, 1100],
    "Engineering": [5000, 5100, 4900]
}, index=["Jan", "Feb", "Mar"])

The Default (`axis="columns"` or `axis=1`)

In [140]:
dept_baselines = pd.Series({"Marketing": 1000, "Engineering": 5000})

print(expenses.sub(dept_baselines)) # subtracts 1000 from all Marketing rows, and 5000 from all Engineering rows

     Marketing  Engineering
Jan          0            0
Feb        200          100
Mar        100         -100


Using `axis="index"` (or `axis=0`)

In [141]:
monthly_rent = pd.Series({"Jan": 200, "Feb": 200, "Mar": 250})

# Matches 'Jan' to 'Jan', then subtracts across both Marketing and Engineering
print(expenses.sub(monthly_rent, axis="index"))

     Marketing  Engineering
Jan        800         4800
Feb       1000         4900
Mar        850         4650
